# Rhea — Expert-Curated Biochemical Reactions Database

**Rhea** is an expert-curated knowledgebase of biochemical reactions developed at the Swiss Institute of Bioinformatics (SIB). It uses chemical entities from the ChEBI (Chemical Entities of Biological Interest) ontology to describe reaction participants precisely. Rhea is the standard reference for reaction annotation in UniProtKB.

| Property | Value |
|---|---|
| URL | https://www.rhea-db.org |
| Reactions | ~15,000+ |
| Compound vocabulary | ChEBI ontology |
| Reaction types | Transport, catalytic, spontaneous |
| Cross-references | UniProtKB, Reactome, MetaCyc |

In [ ]:
import io
import time
from pathlib import Path

import requests
import polars as pl

# TODO

* [x] **Ingest data**
    * [x] Connect to Rhea REST API and confirm access
    * [x] Search and retrieve reactions via the Rhea search API (TSV format)
    * [x] Download UniProt-to-Rhea mapping file from FTP with caching
    * [x] Download Rhea-to-Reactome mapping file from FTP with caching
    * [x] Parse all files into Polars DataFrames with correct dtypes
    * [x] Save to `data/` with caching
* [ ] **Explore and clean**
    * [ ] Summarise reaction counts by direction (master, L-to-R, R-to-L, undirected)
    * [ ] Identify reactions with most participants
    * [ ] Check cross-reference coverage (% with UniProt, Reactome, MetaCyc annotations)
* [ ] **Analysis**
    * [ ] Compute participant-count distribution per reaction
    * [ ] Map enzyme classes (EC numbers) to reaction types
    * [ ] Identify most-connected ChEBI compounds across reactions
* [ ] **Visualization**
    * [ ] Bar chart of reaction types / transport vs. biochemical
    * [ ] Network graph of compound co-occurrence in reactions
* [ ] **Statistical analysis**
    * [ ] Test distribution of reaction sizes
    * [ ] Compare cross-reference density across databases

## 1. Ingest Data

### 1.1 Connect to Rhea REST API

In [ ]:
RHEA_BASE = "https://www.rhea-db.org/rhea"
RHEA_FTP  = "https://ftp.expasy.org/databases/rhea/tsv"

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Test connectivity: search for glucose reactions, return the first 10 rows as TSV
params = {
    "query": "glucose",
    "columns": "rhea-id,equation",
    "format": "tsv",
    "limit": "10",
}
resp = requests.get(RHEA_BASE, params=params, timeout=30)
resp.raise_for_status()

# Parse the TSV snippet with Polars
sample = pl.read_csv(io.StringIO(resp.text), separator="\t")
print(f"API reachable — {len(sample)} rows returned")
print(sample)

### 1.2 Download UniProt-to-Rhea Mapping

In [ ]:
# Columns (tab-separated): RHEA_ID  DIRECTION  MASTER_ID  UNIPROT_ID
# DIRECTION values: UN (undirected master), LR (left-to-right), RL (right-to-left), BI (bidirectional)
UNIPROT_URL  = f"{RHEA_FTP}/rhea2uniprot_sprot.tsv"
UNIPROT_PATH = DATA_DIR / "rhea2uniprot_sprot.tsv"

if not UNIPROT_PATH.exists():
    print(f"Downloading {UNIPROT_PATH.name} ...")
    with requests.get(UNIPROT_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(UNIPROT_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    print(f"Saved to {UNIPROT_PATH}")
else:
    print(f"Already downloaded: {UNIPROT_PATH}")

# Quick preview
preview = pl.read_csv(UNIPROT_PATH, separator="\t", n_rows=5)
print(preview)

### 1.3 Download Rhea-to-Reactome Mapping

In [ ]:
# Columns (tab-separated): RHEA_ID  RHEA_DIRECTION_ID  REACTOME_ID  REACTOME_PATHWAY_NAME
REACTOME_URL  = f"{RHEA_FTP}/rhea2reactome.tsv"
REACTOME_PATH = DATA_DIR / "rhea2reactome.tsv"

if not REACTOME_PATH.exists():
    print(f"Downloading {REACTOME_PATH.name} ...")
    with requests.get(REACTOME_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(REACTOME_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    print(f"Saved to {REACTOME_PATH}")
else:
    print(f"Already downloaded: {REACTOME_PATH}")

# Quick preview
preview = pl.read_csv(REACTOME_PATH, separator="\t", n_rows=5)
print(preview)

### 1.4 Fetch Reactions via Search API

In [ ]:
REACTIONS_PATH = DATA_DIR / "rhea_reactions.tsv"

if not REACTIONS_PATH.exists():
    print("Fetching reactions from Rhea search API (limit=10000) ...")
    params = {
        "query": "*",
        "columns": "rhea-id,equation,chebi-id",
        "format": "tsv",
        "limit": "10000",
    }
    resp = requests.get(RHEA_BASE, params=params, timeout=120)
    resp.raise_for_status()
    REACTIONS_PATH.write_text(resp.text, encoding="utf-8")
    print(f"Saved to {REACTIONS_PATH}")
else:
    print(f"Already downloaded: {REACTIONS_PATH}")

# Parse the TSV response with Polars
reactions_raw = pl.read_csv(
    io.StringIO(REACTIONS_PATH.read_text(encoding="utf-8")),
    separator="\t",
)
print(f"Shape: {reactions_raw.shape}")
reactions_raw.head(5)

### 1.5 Parse into Polars DataFrames

In [ ]:
# ── Reactions ─────────────────────────────────────────────────────────────────
# The search API returns: Rhea ID (integer), Equation (string), ChEBI ID (string,
# may be a semicolon-joined list when multiple ChEBI IDs are reported).
reactions = reactions_raw.rename({c: c.lower().replace(" ", "_").replace("-", "_")
                                   for c in reactions_raw.columns})

# Rhea IDs are plain integers; cast from the default str/i64 to Int32 to save memory.
if reactions["rhea_id"].dtype != pl.Int32:
    reactions = reactions.with_columns(pl.col("rhea_id").cast(pl.Int32))

print("reactions")
print(f"  shape  : {reactions.shape}")
print(f"  dtypes : {dict(zip(reactions.columns, reactions.dtypes))}")
print(reactions.head(5))

# ── UniProt mapping ───────────────────────────────────────────────────────────
# Columns: RHEA_ID  DIRECTION  MASTER_ID  UNIPROT_ID
uniprot_map = pl.read_csv(
    UNIPROT_PATH,
    separator="\t",
    schema_overrides={"RHEA_ID": pl.Int32, "MASTER_ID": pl.Int32},
)
uniprot_map = uniprot_map.with_columns(
    pl.col("DIRECTION").cast(pl.Categorical)
)

print("\nuniprot_map")
print(f"  shape  : {uniprot_map.shape}")
print(f"  dtypes : {dict(zip(uniprot_map.columns, uniprot_map.dtypes))}")
print(uniprot_map.head(5))

# ── Reactome mapping ──────────────────────────────────────────────────────────
# Columns: RHEA_ID  RHEA_DIRECTION_ID  REACTOME_ID  REACTOME_PATHWAY_NAME
reactome_map = pl.read_csv(
    REACTOME_PATH,
    separator="\t",
    schema_overrides={"RHEA_ID": pl.Int32, "RHEA_DIRECTION_ID": pl.Int32},
)

print("\nreactome_map")
print(f"  shape  : {reactome_map.shape}")
print(f"  dtypes : {dict(zip(reactome_map.columns, reactome_map.dtypes))}")
print(reactome_map.head(5))